# Long-Context Evaluation

Advertised context length != usable

## Problem Definition

Context-capcaity gap. Spec sheets say 1M or 10M, Reality says 60%-70% of that is usable, and "usable" depends on the task

## Basic Concept

### Needle-in-a-Haystack (NIAH)

Place a fact at a controlled depth in a long context. Ask the model to retrieve it.

### RULER

13 task types across 4 categories: retrieval(single/multi-key/multi-value), mult-hop tracing ...

### LongBench

503 muliple-choice questions, 8k - 2M word contexts.



# Build your Own

## Cutom NIAH

In [4]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter, load_project_env

load_project_env()

def tokenize(text):
    # Just fake it
    return text.split()

def build_haystack(filler_text, needle, depth_ratio, total_tokens):
    if not (0.0 <= depth_ratio <= 1.0):
        raise ValueError("deep_ratio must be between 0.0 and 1.0")
    if not (0 < total_tokens <= 100000):
        raise ValueError("total_tokens must be between 0 and 100000")
    
    # Calculate the number of tokens for the filler text
    filler_tokens = tokenize(filler_text)
    needle_tokens = tokenize(needle)

    if not filler_tokens:
        raise ValueError("filler_text must contain at least one token")

    body_len = max(total_tokens - len(needle_tokens), 0)
    while len(filler_tokens) < body_len:
        filler_tokens += filler_tokens
    filler_tokens = filler_tokens[:body_len]

    insert_at = min(int(body_len * depth_ratio), body_len)
    haystack = filler_tokens[:insert_at] + needle_tokens + filler_tokens[insert_at:]
    return " ".join(haystack)


def score_niah(model, haystack, question, expected):
    answer = model.invoke(f"Context: {haystack}\nQuestion: {question}\nAnswer:")
    print(answer.content)
    print(expected)
    return 1 if answer.content.lower() in expected.lower() else 0

from langchain.chat_models import init_chat_model
model = init_chat_model("deepseek:deepseek-chat", extra_body={"thinking": {"type": "disabled"}})

with SectionPrinter("NIAH"):
    haystack = build_haystack("This is a filler text" * 30, "The answer is 42", 0.5, 1000)
    print(score_niah(model, haystack, "What is the answer to the universe?", "42"))




============================NIAH============================
42
42
1


## A multi-needle variant

In [ ]:
def build_multi_needle(filler_text, needles, total_tokens):
    depths = [0.1, 0.4, 0.7]
    chunks = [filler_text[:(int(total_tokens * 0.1))]]

    for depth, needle in zip(depths, needles):
        chunks.append(needle)
        next_chunk = filler_text[int(total_tokens * depth): int(total_tokens * (depth + 0.3))]
        chunks.append(next_chunk)
    return " ".join(chunks)

with SectionPrinter("Multi-needle"):
    haystack = build_multi_needle("This is a filler text" * 30, ["A = 1", "B = 2", " C = A + B"], 1000)
    print(haystack)

    question = "What is the value of C?"
    answer = model.invoke(f"Context: {haystack}\nQuestion: {question}\nAnswer:")
    print(answer.content)


========================Multi-needle========================
This is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler The answer is 42  textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textT The answer is 43 his is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler textThis is a filler text The answer is 44 
